# 03 — Обучение per-user скорера

Вся логика — в `research/scripts/train_scorer.py` и `research/configs/scorer_50m.yaml`:
препроцессинг, обучение SASRec, sanity на test и кэш топ-K скоров.

Смок локально: `--config research/configs/smoke_local.yaml`.
Пересчитать только кэш поверх готового чекпоинта: `--cache-only`.

In [ ]:
# Colab: раскомментировать. Локально ячейка не нужна.
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -q https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q uv && uv pip install --system -e ".[research]"


## Запуск

В `artifacts/gsasrec/` появятся `best.pt`, `item_id_to_idx.pkl`, `metrics.csv`, `config.resolved.json`, `run.json`; кэш — в `artifacts/user_scores_cache/scores.parquet`.

In [ ]:
CONFIG = 'research/configs/scorer_50m.yaml'

!uv run python research/scripts/train_scorer.py --config {CONFIG}


## Кривая обучения

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ARTIFACTS = Path.cwd() / 'artifacts'
metrics = pd.read_csv(ARTIFACTS / 'gsasrec' / 'metrics.csv')

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.plot(metrics['epoch'], metrics['train_loss'], 'b-o', label='train loss')
ax2.plot(metrics['epoch'], metrics['val_ndcg@10'], 'r-s', label='val NDCG@10')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss', color='b')
ax2.set_ylabel('val NDCG@10', color='r')
ax1.grid(alpha=0.3); plt.title('SASRec training'); plt.tight_layout(); plt.show()
metrics.tail()


## Проверка кэша скоров

In [ ]:
scores = pd.read_parquet(ARTIFACTS / 'user_scores_cache' / 'scores.parquet')
print('rows:', len(scores), '| users:', scores.uid.nunique(),
      '| K:', scores.groupby('uid').size().unique())
scores.head()
